# 22. AI for Data Science

고정 split, metric, 예산 안에서 baseline과 시계열 forecast를 비교하고 실험 기록을 재현합니다.

핵심 실습은 NumPy와 Python 표준 라이브러리만 사용합니다. API 호출, 모델 다운로드, GPU는 사용하지 않습니다. 수치는 고정된 작은 입력의 계산이며 실제 pretrained model의 성능이 아닙니다.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(27)

## Tabular baseline

고정 seed로 만든 synthetic regression 문제입니다. 훈련 데이터만으로 mean과 scale을 계산합니다.

In [2]:
import hashlib,json
X=rng.normal(size=(100,3))
y=2*X[:,0]-.5*X[:,1]+rng.normal(scale=.2,size=100)
train=np.arange(60);val=np.arange(60,80);test=np.arange(80,100)
mu=X[train].mean(axis=0);sd=X[train].std(axis=0)
Z=(X-mu)/sd
def design(z):return np.column_stack([np.ones(len(z)),z])
beta=np.linalg.lstsq(design(Z[train]),y[train],rcond=None)[0]
pred=design(Z)@beta
mae=lambda a,b:float(np.abs(a-b).mean())
print("validation MAE:",mae(pred[val],y[val]))
assert set(train).isdisjoint(val) and set(val).isdisjoint(test)

validation MAE: 0.19636027955023


## 실험 registry

후보는 validation만으로 고릅니다. 이 예의 ridge solver는 intercept를 penalty에서 제외합니다.

In [3]:
registry=[]
for alpha in [0.,.1,1.,10.]:
    A=design(Z[train]);penalty=np.eye(A.shape[1]);penalty[0,0]=0
    b=np.linalg.solve(A.T@A+alpha*penalty,A.T@y[train])
    score=mae(design(Z[val])@b,y[val])
    registry.append({"alpha":alpha,"validation_mae":score,"beta":b})
winner=min(registry,key=lambda r:r["validation_mae"])
print([{k:v for k,v in r.items()if k!="beta"}for r in registry])
print("chosen alpha:",winner["alpha"])
# Test is accessed only after the choice is frozen.
final_test_mae=mae(design(Z[test])@winner["beta"],y[test])
print("final test MAE:",final_test_mae)

[{'alpha': 0.0, 'validation_mae': 0.19636027955023}, {'alpha': 0.1, 'validation_mae': 0.19624414942725663}, {'alpha': 1.0, 'validation_mae': 0.19521256631433037}, {'alpha': 10.0, 'validation_mae': 0.31589703513641315}]
chosen alpha: 1.0
final test MAE: 0.15884095840352877


## 시간 순서와 seasonal naive

주기가4인 작은 series에 trend를 더합니다. 미래 actual을 feature로 쓰는 oracle은 허용되는 baseline이 아닙니다.

In [4]:
t=np.arange(40)
series=np.tile([2.,4.,3.,1.],10)+.05*t
def seasonal_naive(history,horizon,period):
    if len(history)<period:raise ValueError("insufficient history")
    return np.tile(history[-period:],int(np.ceil(horizon/period)))[:horizon]
rolling=[]
for origin in [20,24,28,32]:
    prediction=seasonal_naive(series[:origin],4,4)
    rolling.append(mae(prediction,series[origin:origin+4]))
print("rolling-origin MAE:",rolling)
assert np.allclose(rolling,.2)

rolling-origin MAE: [0.20000000000000007, 0.20000000000000007, 0.20000000000000007, 0.20000000000000007]


## Shift failure

새 regime의 수준이 올라가면 기존 history를 반복하는 예측이 실패합니다.

In [5]:
shifted=series.copy();shifted[32:]+=5
origin=32
normal_pred=seasonal_naive(series[:origin],4,4)
shift_pred=seasonal_naive(shifted[:origin],4,4)
normal_error=mae(normal_pred,series[origin:origin+4])
shift_error=mae(shift_pred,shifted[origin:origin+4])
print("normal and shifted errors:",normal_error,shift_error)
assert shift_error>normal_error

normal and shifted errors: 0.20000000000000007 5.199999999999999


## 과제

- 후보실험 예산을2로 제한하여 registry에 실행한 후보만 남기세요. 선택 결과가 달라질 수 있는지 확인하세요.
- code·data fingerprint·seed·split·metric을 포함하는 재현 기록을 만들고 JSON serialization을 확인하세요.

In [6]:
# 과제 풀이 1
budget=2
executed=registry[:budget]
limited_winner=min(executed,key=lambda r:r["validation_mae"])
assert len(executed)==budget
print("limited winner:",limited_winner["alpha"])
# 과제 풀이 2
fingerprint=hashlib.sha256(np.ascontiguousarray(X).tobytes()).hexdigest()
record={"seed":27,"data_sha256":fingerprint,"split":[60,20,20],
        "metric":"MAE","chosen_alpha":float(winner["alpha"]),
        "validation_mae":float(winner["validation_mae"]),"test_mae":final_test_mae,
        "code":"closed-form-ridge-v1"}
serialized=json.dumps(record,sort_keys=True)
assert json.loads(serialized)["metric"]=="MAE"
print(serialized)

limited winner: 0.1
{"chosen_alpha": 1.0, "code": "closed-form-ridge-v1", "data_sha256": "111e579ecff23ee11ac179ae87cf62e158c99bf4b79bd97bed047ef6f2635345", "metric": "MAE", "seed": 27, "split": [60, 20, 20], "test_mae": 0.15884095840352877, "validation_mae": 0.19521256631433037}


## Foundation model은 어디에 연결하나

실제 TabPFN·TimesFM·Chronos·Moirai를 비교한다면 위 split·metric·forecast origin을 유지하고 candidate 함수만 교체합니다. 이 notebook은 모델 weights를 받거나 pretrained 성능을 재현하지 않습니다. 지원 범위와 라이선스는 공식 구현에서 확인합니다. [TabPFN](https://arxiv.org/abs/2207.01848), [TimesFM](https://github.com/google-research/timesfm), [Chronos](https://github.com/amazon-science/chronos-forecasting), [Moirai](https://github.com/SalesforceAIResearch/uni2ts).

## Full Model Lab
The lecture HTML contains an inline numerical demo and a Model Lab panel. These notebooks run the corresponding pretrained models in a separate Python environment.
- [22_ai-for-ds.ipynb](model-labs/22_ai-for-ds.ipynb)
